In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [2]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler


In [3]:
 # LOAD DATASETS
# =====================================================

air_df = pd.read_csv("air_quality.csv")
heart_df = pd.read_csv("heart.csv")


In [4]:
print("========== AIR QUALITY DATASET ==========")
print(air_df.head())

print("\n========== HEART DISEASE DATASET ==========")
print(heart_df.head())

========== AIR QUALITY DATASET ==========
        City        Date  PM2.5  PM10     NO    NO2    NOx  NH3     CO    SO2  \
0  Ahmedabad  2015-01-01    NaN   NaN   0.92  18.22  17.15  NaN   0.92  27.64   
1  Ahmedabad  2015-01-02    NaN   NaN   0.97  15.69  16.46  NaN   0.97  24.55   
2  Ahmedabad  2015-01-03    NaN   NaN  17.40  19.30  29.70  NaN  17.40  29.07   
3  Ahmedabad  2015-01-04    NaN   NaN   1.70  18.48  17.97  NaN   1.70  18.59   
4  Ahmedabad  2015-01-05    NaN   NaN  22.10  21.42  37.76  NaN  22.10  39.33   

       O3  Benzene  Toluene  Xylene  AQI AQI_Bucket  
0  133.36     0.00     0.02    0.00  NaN        NaN  
1   34.06     3.68     5.50    3.77  NaN        NaN  
2   30.70     6.80    16.40    2.25  NaN        NaN  
3   36.08     4.43    10.14    1.00  NaN        NaN  
4   39.31     7.01    18.89    2.78  NaN        NaN  

========== HEART DISEASE DATASET ==========
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0      

In [5]:
# =====================================================
# a. DATA CLEANING
# =====================================================

print("\n========== DATA CLEANING ==========")

# Check missing values
print("\nMissing Values in Air Quality Dataset:")
print(air_df.isnull().sum())

print("\nMissing Values in Heart Dataset:")
print(heart_df.isnull().sum())

# Fill missing numeric values with mean
air_df.fillna(air_df.mean(numeric_only=True), inplace=True)
heart_df.fillna(heart_df.mean(numeric_only=True), inplace=True)

# Remove duplicate rows
air_df.drop_duplicates(inplace=True)
heart_df.drop_duplicates(inplace=True)

print("\nData Cleaning Completed")


========== DATA CLEANING ==========

Missing Values in Air Quality Dataset:
City              0
Date              0
PM2.5          4598
PM10          11140
NO             3582
NO2            3585
NOx            4185
NH3           10328
CO             2059
SO2            3854
O3             4022
Benzene        5623
Toluene        8041
Xylene        18109
AQI            4681
AQI_Bucket     4681
dtype: int64

Missing Values in Heart Dataset:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

Data Cleaning Completed


In [6]:
# =====================================================
# b. DATA INTEGRATION
# =====================================================

print("\n========== DATA INTEGRATION ==========")

# Create common ID column
air_df['ID'] = range(1, len(air_df)+1)
heart_df['ID'] = range(1, len(heart_df)+1)

# Merge datasets using ID
merged_df = pd.merge(air_df, heart_df, on='ID')

print("\nMerged Dataset:")
print(merged_df.head())


========== DATA INTEGRATION ==========

Merged Dataset:
        City        Date      PM2.5        PM10     NO    NO2    NOx  \
0  Ahmedabad  2015-01-01  67.450578  118.127103   0.92  18.22  17.15   
1  Ahmedabad  2015-01-02  67.450578  118.127103   0.97  15.69  16.46   
2  Ahmedabad  2015-01-03  67.450578  118.127103  17.40  19.30  29.70   
3  Ahmedabad  2015-01-04  67.450578  118.127103   1.70  18.48  17.97   
4  Ahmedabad  2015-01-05  67.450578  118.127103  22.10  21.42  37.76   

         NH3     CO    SO2  ...  chol  fbs  restecg  thalach  exang oldpeak  \
0  23.483476   0.92  27.64  ...   212    0        1      168      0     1.0   
1  23.483476   0.97  24.55  ...   203    1        0      155      1     3.1   
2  23.483476  17.40  29.07  ...   174    0        1      125      1     2.6   
3  23.483476   1.70  18.59  ...   203    0        1      161      0     0.0   
4  23.483476  22.10  39.33  ...   294    1        1      106      0     1.9   

   slope  ca  thal  target  
0     

In [7]:
# =====================================================
# c. DATA TRANSFORMATION
# =====================================================

print("\n========== DATA TRANSFORMATION ==========")

# Label Encoding for categorical columns
le = LabelEncoder()

for col in heart_df.columns:
    if heart_df[col].dtype == 'object':
        heart_df[col] = le.fit_transform(heart_df[col])

print("\nTransformed Heart Dataset:")
print(heart_df.head())



========== DATA TRANSFORMATION ==========

Transformed Heart Dataset:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   
3   61    1   0       148   203    0        1      161      0      0.0      2   
4   62    0   0       138   294    1        1      106      0      1.9      1   

   ca  thal  target  ID  
0   2     3       0   1  
1   0     3       0   2  
2   0     3       0   3  
3   1     3       0   4  
4   3     2       0   5  


In [8]:
# =====================================================
# d. ERROR CORRECTING
# =====================================================

print("\n========== ERROR CORRECTING ==========")

# Correct negative values if present

numeric_cols = air_df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
    air_df[col] = air_df[col].apply(lambda x: abs(x))

print("\nErrors Corrected Successfully")



========== ERROR CORRECTING ==========

Errors Corrected Successfully


In [9]:
# =====================================================
# e. DATA MODEL BUILDING
# =====================================================

print("\n========== DATA MODEL BUILDING ==========")

# Separate Features and Target
X = heart_df.drop('target', axis=1)
y = heart_df['target']

# Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


========== DATA MODEL BUILDING ==========


In [10]:
# =====================================================
# FEATURE SCALING (Fixes Convergence Warning)
# =====================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [11]:
# =====================================================
# LOGISTIC REGRESSION MODEL
# =====================================================

model = LogisticRegression(max_iter=5000)

# Train Model
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("\nModel Accuracy:")
print(accuracy)


Model Accuracy:
0.7540983606557377
